# Epi Info AI algorithm validation lab — V0.4

This executable specification derives a 2 × 2 table from the 96-record synthetic foodborne-outbreak corpus, then compares the deployed Rust/WebAssembly kernel—including exact tails, the conditional-MLE odds ratio, and exact confidence limits—with hand-checkable anchors and independent SciPy calculations.

> **Validation state:** `epi.table2x2` is a candidate operation. A passing notebook is evidence, not statistical approval; committed fixtures, GitLab CI, legacy comparison, and independent review remain authoritative.


## 1. Load and verify the frozen corpus

The fixture records the exact source hash and derivation semantics. The notebook never rewrites the CSV.


In [ ]:
import csv, hashlib, io, math, platform, sys
from pyodide.http import pyfetch

FIXTURE_URL = '/validation-fixtures/foodborne-outbreak-v1-table2x2.json'
DATA_URL = '/examples/foodborne-outbreak-investigation.csv'
WASM_URL = '/epi2x2.wasm'
fixture_response = await pyfetch(FIXTURE_URL)
fixture_response.raise_for_status()
fixture = await fixture_response.json()
data_response = await pyfetch(DATA_URL)
data_response.raise_for_status()
data_bytes = await data_response.bytes()
data_hash = hashlib.sha256(data_bytes).hexdigest()
assert data_hash == fixture['dataset']['sha256']
rows = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
assert len(rows) == fixture['dataset']['rows']
{'rows': len(rows), 'sha256': data_hash}


## 2. Derive the potato-salad table

Cases are Confirmed, Probable, or Suspected. `Yes` and `y` mean exposed; `No` and `n` mean unexposed. Required-input missing values are excluded and counted.


In [ ]:
rules = fixture['derivation']
case_values = {value.casefold() for value in rules['caseValues']}
noncase_values = {value.casefold() for value in rules['noncaseValues']}
yes_values = {value.casefold() for value in rules['yesValues']}
no_values = {value.casefold() for value in rules['noValues']}
a = b = c = d = excluded = 0
for row in rows:
    status = row[rules['caseField']].strip().casefold()
    exposure = row[rules['exposureField']].strip().casefold()
    if status not in case_values | noncase_values or exposure not in yes_values | no_values:
        excluded += 1
        continue
    is_case, is_exposed = status in case_values, exposure in yes_values
    if is_exposed and is_case: a += 1
    elif is_exposed: b += 1
    elif is_case: c += 1
    else: d += 1
derived = {'exposedCases': a, 'exposedNonCases': b, 'unexposedCases': c, 'unexposedNonCases': d}
assert derived == {key: fixture['input'][key] for key in derived}
assert excluded == rules['excludedRows']
{'table': [[a, b], [c, d]], 'included': a + b + c + d, 'excluded': excluded}


## 3. Call the deployed Rust/WASM release binary

Pyodide's JavaScript bridge calls the same WASM exports used by the browser application. Rust is compiled ahead of time; it is not a second notebook kernel.


In [ ]:
from js import WebAssembly, fetch
wasm_response = await fetch(WASM_URL)
if not wasm_response.ok: raise RuntimeError(f'Unable to load WASM: HTTP {wasm_response.status}')
wasm_module = await WebAssembly.instantiate(await wasm_response.arrayBuffer())
rust = wasm_module.instance.exports
z = 1.959963984540054
rust_result = {
    'riskExposed': float(rust.risk_exposed(a, b)),
    'riskUnexposed': float(rust.risk_unexposed(c, d)),
    'riskRatio': float(rust.risk_ratio(a, b, c, d)),
    'riskRatioLower': float(rust.risk_ratio_ci_lower(a, b, c, d, z)),
    'riskRatioUpper': float(rust.risk_ratio_ci_upper(a, b, c, d, z)),
    'oddsRatio': float(rust.odds_ratio(a, b, c, d)),
    'oddsRatioLower': float(rust.odds_ratio_ci_lower(a, b, c, d, z)),
    'oddsRatioUpper': float(rust.odds_ratio_ci_upper(a, b, c, d, z)),
    'riskDifference': float(rust.risk_difference(a, b, c, d)),
    'riskDifferenceLower': float(rust.risk_difference_ci_lower(a, b, c, d, z)),
    'riskDifferenceUpper': float(rust.risk_difference_ci_upper(a, b, c, d, z)),
    'fisherLeft': float(rust.fisher_exact_left(a, b, c, d)),
    'fisherRight': float(rust.fisher_exact_right(a, b, c, d)),
    'fisherOneTailed': float(rust.fisher_exact_one_tailed(a, b, c, d)),
    'fisherTwoTailed': float(rust.fisher_exact_two_tailed(a, b, c, d)),
    'midPLeft': float(rust.mid_p_exact_left(a, b, c, d)),
    'midPRight': float(rust.mid_p_exact_right(a, b, c, d)),
    'midPOneTailed': float(rust.mid_p_exact_one_tailed(a, b, c, d)),
    'conditionalOddsRatio': float(rust.conditional_odds_ratio(a, b, c, d)),
    'conditionalOddsRatioFisherLower': float(rust.conditional_odds_ratio_fisher_lower(a, b, c, d, fixture['input']['confidenceLevel'])),
    'conditionalOddsRatioFisherUpper': float(rust.conditional_odds_ratio_fisher_upper(a, b, c, d, fixture['input']['confidenceLevel'])),
    'conditionalOddsRatioMidPLower': float(rust.conditional_odds_ratio_mid_p_lower(a, b, c, d, fixture['input']['confidenceLevel'])),
    'conditionalOddsRatioMidPUpper': float(rust.conditional_odds_ratio_mid_p_upper(a, b, c, d, fixture['input']['confidenceLevel'])),
}
for name, function in {'pearson': rust.pearson_chi_square, 'mantelHaenszel': rust.mantel_haenszel_chi_square, 'yates': rust.yates_chi_square}.items():
    statistic = float(function(a, b, c, d))
    rust_result[f'{name}ChiSquare'] = statistic
    rust_result[f'{name}PValue'] = float(rust.chi_square_p_value(statistic))
rust_result


## 4. Independent Python reference

SciPy supplies the confidence multiplier, Katz risk-ratio interval, chi-square survival function, fixed-margin hypergeometric distribution, and conditional noncentral-hypergeometric reference. The named Wald and exact-tail equations remain visible for audit.


In [ ]:
import scipy
from scipy.optimize import brentq
from scipy.stats import chi2, hypergeom, nchypergeom_fisher, norm
from scipy.stats.contingency import odds_ratio as scipy_odds_ratio, relative_risk
confidence_level = fixture['input']['confidenceLevel']
python_z = float(norm.ppf(0.5 + confidence_level / 2))
rr_reference = relative_risk(a, a + b, c, c + d)
rr_interval = rr_reference.confidence_interval(confidence_level)
risk_exposed, risk_unexposed = a / (a + b), c / (c + d)
odds_ratio = a * d / (b * c)
or_se = math.sqrt(1/a + 1/b + 1/c + 1/d)
risk_difference = risk_exposed - risk_unexposed
rd_se = math.sqrt(risk_exposed*(1-risk_exposed)/(a+b) + risk_unexposed*(1-risk_unexposed)/(c+d))
total, row_one, column_one = a+b+c+d, a+b, a+c
support = range(max(0, row_one-(total-column_one)), min(row_one, column_one)+1)
observed_probability = float(hypergeom.pmf(a, total, column_one, row_one))
fisher_left = float(hypergeom.cdf(a, total, column_one, row_one))
fisher_right = float(hypergeom.sf(a-1, total, column_one, row_one))
fisher_two = sum(float(hypergeom.pmf(x, total, column_one, row_one)) for x in support if hypergeom.pmf(x, total, column_one, row_one) <= observed_probability * 1.000001)
conditional_reference = scipy_odds_ratio([[a, b], [c, d]], kind='conditional')
conditional_interval = conditional_reference.confidence_interval(confidence_level)
alpha_half = (1-confidence_level)/2
def noncentral(log_odds): return nchypergeom_fisher(total, column_one, row_one, math.exp(log_odds))
mid_p_lower = math.exp(brentq(lambda eta: noncentral(eta).sf(a) + 0.5*noncentral(eta).pmf(a) - alpha_half, -50, 50))
mid_p_upper = math.exp(brentq(lambda eta: noncentral(eta).cdf(a-1) + 0.5*noncentral(eta).pmf(a) - alpha_half, -50, 50))
python_result = {
    'riskExposed': risk_exposed, 'riskUnexposed': risk_unexposed,
    'riskRatio': float(rr_reference.relative_risk), 'riskRatioLower': float(rr_interval.low), 'riskRatioUpper': float(rr_interval.high),
    'oddsRatio': odds_ratio, 'oddsRatioLower': math.exp(math.log(odds_ratio)-python_z*or_se), 'oddsRatioUpper': math.exp(math.log(odds_ratio)+python_z*or_se),
    'riskDifference': risk_difference, 'riskDifferenceLower': risk_difference-python_z*rd_se, 'riskDifferenceUpper': risk_difference+python_z*rd_se,
    'fisherLeft': fisher_left, 'fisherRight': fisher_right, 'fisherOneTailed': min(fisher_left, fisher_right), 'fisherTwoTailed': fisher_two,
    'midPLeft': fisher_left-observed_probability/2, 'midPRight': fisher_right-observed_probability/2, 'midPOneTailed': min(fisher_left, fisher_right)-observed_probability/2,
    'conditionalOddsRatio': float(conditional_reference.statistic), 'conditionalOddsRatioFisherLower': float(conditional_interval.low), 'conditionalOddsRatioFisherUpper': float(conditional_interval.high),
    'conditionalOddsRatioMidPLower': mid_p_lower, 'conditionalOddsRatioMidPUpper': mid_p_upper,
}
for name in ('pearson', 'mantelHaenszel', 'yates'):
    statistic = rust_result[f'{name}ChiSquare']
    python_result[f'{name}ChiSquare'] = statistic
    python_result[f'{name}PValue'] = float(chi2.sf(statistic, df=1))
python_result


## 5. Compare Rust/WASM, Python, and the candidate golden


In [ ]:
import pandas as pd
expected = fixture['expected']
golden = {
    'riskExposed': expected['riskExposed'], 'riskUnexposed': expected['riskUnexposed'],
    'riskRatio': expected['riskRatio'], 'riskRatioLower': expected['riskRatioConfidenceInterval']['lower'], 'riskRatioUpper': expected['riskRatioConfidenceInterval']['upper'],
    'oddsRatio': expected['oddsRatio'], 'oddsRatioLower': expected['oddsRatioConfidenceInterval']['lower'], 'oddsRatioUpper': expected['oddsRatioConfidenceInterval']['upper'],
    'riskDifference': expected['riskDifference'], 'riskDifferenceLower': expected['riskDifferenceConfidenceInterval']['lower'], 'riskDifferenceUpper': expected['riskDifferenceConfidenceInterval']['upper'],
    'fisherLeft': expected['fisherExact']['left'], 'fisherRight': expected['fisherExact']['right'], 'fisherOneTailed': expected['fisherExact']['oneTailed'], 'fisherTwoTailed': expected['fisherExact']['twoTailed'],
    'midPLeft': expected['midPExact']['left'], 'midPRight': expected['midPExact']['right'], 'midPOneTailed': expected['midPExact']['oneTailed'],
    'conditionalOddsRatio': expected['conditionalOddsRatio']['estimate'], 'conditionalOddsRatioFisherLower': expected['conditionalOddsRatio']['fisherConfidenceInterval']['lower'], 'conditionalOddsRatioFisherUpper': expected['conditionalOddsRatio']['fisherConfidenceInterval']['upper'],
    'conditionalOddsRatioMidPLower': expected['conditionalOddsRatio']['midPConfidenceInterval']['lower'], 'conditionalOddsRatioMidPUpper': expected['conditionalOddsRatio']['midPConfidenceInterval']['upper'],
}
for name in ('pearson', 'mantelHaenszel', 'yates'):
    golden[f'{name}ChiSquare'] = expected[f'{name}ChiSquare']
    golden[f'{name}PValue'] = expected[f'{name}PValue']
absoluteTolerance = float(fixture['comparisons']['finiteResults']['tolerance'])
conditionalTolerance = float(fixture['comparisons']['conditionalOddsRatioResults']['tolerance'])
records = []
for measure in golden:
    for reference_name, reference in (('candidate golden', golden), ('Python', python_result)):
        difference = abs(rust_result[measure] - reference[measure])
        tolerance = conditionalTolerance if measure.startswith('conditionalOddsRatio') else absoluteTolerance
        records.append({'measure': measure, 'comparison': f'Rust/WASM vs {reference_name}', 'rust_wasm': rust_result[measure], 'reference': reference[measure], 'absolute_difference': difference, 'absoluteTolerance': tolerance, 'passed': math.isclose(rust_result[measure], reference[measure], rel_tol=0, abs_tol=tolerance)})
comparison = pd.DataFrame(records)
assert comparison['passed'].all(), comparison.loc[~comparison['passed']]
comparison


## 6. Provenance

The committed 100-case legacy-derived corpus is exercised in CI for Fisher p-values and exact confidence limits. Independent approval and broader pathological evidence remain explicitly open.


In [ ]:
wasm_hash_response = await pyfetch(WASM_URL)
wasm_hash_response.raise_for_status()
wasm_sha256 = hashlib.sha256(await wasm_hash_response.bytes()).hexdigest()
provenance = {
    'fixture_id': fixture['id'], 'dataset_sha256': data_hash,
    'operation': fixture['operation']['id'], 'result_schema_version': fixture['operation']['resultSchemaVersion'],
    'engine_id': fixture['operation']['engineId'], 'engine_version': fixture['operation']['engineVersion'],
    'wasm_sha256': wasm_sha256, 'python': sys.version.split()[0], 'python_platform': platform.platform(),
    'scipy': scipy.__version__, 'methods': fixture['methods'], 'legacy_reference': fixture['review']['legacyReference'],
    'candidate_python_reference': fixture['review']['independentPythonReference'],
    'all_comparisons_passed': bool(comparison['passed'].all()),
}
provenance


## Next validation increment

Add independently reviewed extreme-margin and convergence cases, benchmark exact operations for Worker routing, and preserve every discrepancy as reviewable evidence before promotion.
